# 两票候选名称：大语言模型语义筛选

本 Notebook 只处理 `name_two_vote_candidates.csv` 中票数恰好为 2 的候选名称。

模型仅根据候选名称的字面语义，将其分类为：`具体店名`、`平台/线上商家名`、`泛称或句子片段`、`不确定`。它**不**根据名称直接证明该名称与 `restId` 的真实对应关系。


## 第 1 步：配置路径与试运行范围

首次请保持 `RUN_LIMIT = 20`，人工检查输出后，再改为 `None` 处理全部 2,295 条候选。


In [14]:
from pathlib import Path

PROJECT_DIR = Path(r"D:\携程实习文件\项目2\推荐系统数据集\推荐系统数据集\店家")
RAW_DIR = PROJECT_DIR / "data" / "raw"
DERIVED_DIR = PROJECT_DIR / "data" / "derived"
FINAL_DIR = PROJECT_DIR / "data" / "final"
RATING_SPLIT_DIR = PROJECT_DIR / "data" / "splits" / "rating_split"
NAME_OUTPUT_DIR = PROJECT_DIR / "outputs" / "name_completion"
CATEGORY_OUTPUT_DIR = PROJECT_DIR / "outputs" / "category"
INPUT_PATH = NAME_OUTPUT_DIR / "name_two_vote_candidates.csv"
OUTPUT_PATH = NAME_OUTPUT_DIR / "name_two_vote_llm_screened.csv"

# 首次只审核 20 条；确认结果合理后改为 None。
RUN_LIMIT = None
BATCH_SIZE = 20


## 第 2 步：读取候选并查看数据概览


In [15]:
import csv

with INPUT_PATH.open("r", encoding="utf-8-sig", newline="") as source:
    candidates = list(csv.DictReader(source))

print(f"两票候选总数：{len(candidates):,}")
print("前 5 条：")
for row in candidates[:5]:
    print(row)


两票候选总数：2,295
前 5 条：
{'restId': '11', 'name_candidate': '东百', 'candidate_status': '商家名称候选', 'vote_count': '2'}
{'restId': '12', 'name_candidate': '妇儿', 'candidate_status': '商家名称候选', 'vote_count': '2'}
{'restId': '14', 'name_candidate': '家乐福', 'candidate_status': '商家名称候选', 'vote_count': '2'}
{'restId': '15', 'name_candidate': '同样的百佳超市', 'candidate_status': '商家名称候选', 'vote_count': '2'}
{'restId': '15', 'name_candidate': '市场', 'candidate_status': '商家名称候选', 'vote_count': '2'}


## 第 3 步：选择模型服务并输入 API Key

默认配置为通义千问的 OpenAI 兼容接口。若使用 DeepSeek，请把 `PROVIDER` 改为 `deepseek`，并检查你的账户中可用的模型名称。

API Key 只在当前运行内存中保存；不要写入 Notebook，也不要上传到 GitHub。


In [16]:
# 如尚未安装，请先运行一次本单元格。
!pip install -q -U openai


In [17]:
import getpass
from openai import OpenAI

PROVIDER = "deepseek"  # 可选："qwen" 或 "deepseek"

PROVIDER_SETTINGS = {
    "qwen": {
        "base_url": "https://dashscope.aliyuncs.com/compatible-mode/v1",
        "model": "qwen-plus",
        "key_prompt": "请输入百炼（通义千问）API Key：",
    },
    "deepseek": {
        "base_url": "https://api.deepseek.com",
        "model": "deepseek-v4-flash",  # 名称语义筛选默认使用快速模型；可改为 deepseek-v4-pro
        "key_prompt": "请输入 DeepSeek API Key：",
    },
}

settings = PROVIDER_SETTINGS[PROVIDER]
api_key = getpass.getpass(settings["key_prompt"])
client = OpenAI(api_key=api_key, base_url=settings["base_url"])
MODEL_NAME = settings["model"]

print(f"已配置服务：{PROVIDER}；模型：{MODEL_NAME}")


已配置服务：deepseek；模型：deepseek-v4-flash


## 第 4 步：定义名称语义审核提示词

每批仅发送候选名称、候选状态和票数，不发送评论正文，以控制 token 消耗。


In [18]:
import json
import time

ALLOWED_LABELS = {"具体店名", "平台/线上商家名", "泛称或句子片段", "不确定"}

def classify_batch(rows):
    payload = [
        {
            "restId": row["restId"],
            "name_candidate": row["name_candidate"],
            "candidate_status": row["candidate_status"],
            "vote_count": int(row["vote_count"]),
        }
        for row in rows
    ]

    prompt = f"""
你是中文商家名称数据清洗助手。请只根据候选名称的字面语义分类，不能虚构评论证据。

分类标签只能为：具体店名、平台/线上商家名、泛称或句子片段、不确定。
淘宝、京东、当当、苏宁等交易平台应标为“平台/线上商家名”。
“这家”“附近超市”“一楼是火锅”等泛称或评论片段应标为“泛称或句子片段”。

输入记录：
{json.dumps(payload, ensure_ascii=False)}

只返回 JSON，格式必须为：
{{"results":[
  {{"restId":"", "name_candidate":"", "llm_label":"", "llm_reason":""}}
]}}
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "你只输出符合要求的 JSON。"},
            {"role": "user", "content": prompt},
        ],
        response_format={"type": "json_object"},
        temperature=0,
    )
    result = json.loads(response.choices[0].message.content)["results"]
    if len(result) != len(rows):
        raise ValueError("模型返回的记录数与输入批次不一致，请检查后重试。")
    for item in result:
        if item.get("llm_label") not in ALLOWED_LABELS:
            item["llm_label"] = "不确定"
    return result


## 第 5 步：分批运行并断点续跑

每完成一批会立即写入 CSV。中断后再次运行时，会自动跳过已完成的 `restId + 候选名称`。


In [19]:
def load_finished_keys():
    if not OUTPUT_PATH.exists():
        return set()
    with OUTPUT_PATH.open("r", encoding="utf-8-sig", newline="") as source:
        return {
            (row["restId"], row["name_candidate"])
            for row in csv.DictReader(source)
        }

finished_keys = load_finished_keys()
pending = [
    row for row in candidates
    if (row["restId"], row["name_candidate"]) not in finished_keys
]
if RUN_LIMIT is not None:
    pending = pending[:RUN_LIMIT]

print(f"本次待审核：{len(pending):,} 条；已完成：{len(finished_keys):,} 条")

output_fields = [
    "restId", "name_candidate", "candidate_status", "vote_count",
    "llm_label", "llm_reason",
]
write_header = not OUTPUT_PATH.exists()

for start in range(0, len(pending), BATCH_SIZE):
    batch = pending[start:start + BATCH_SIZE]
    try:
        results = classify_batch(batch)
    except Exception as error:
        print(f"第 {start // BATCH_SIZE + 1} 批失败：{error}")
        print("已完成批次已保存；检查后重新运行本单元格即可续跑。")
        break

    result_map = {(item["restId"], item["name_candidate"]): item for item in results}
    output_rows = []
    for row in batch:
        item = result_map[(row["restId"], row["name_candidate"])]
        output_rows.append({
            **row,
            "llm_label": item["llm_label"],
            "llm_reason": item.get("llm_reason", ""),
        })

    with OUTPUT_PATH.open("a", encoding="utf-8-sig", newline="") as target:
        writer = csv.DictWriter(target, fieldnames=output_fields)
        if write_header:
            writer.writeheader()
            write_header = False
        writer.writerows(output_rows)

    print(f"已完成 {min(start + BATCH_SIZE, len(pending)):,}/{len(pending):,} 条")
    time.sleep(0.5)

print(f"结果文件：{OUTPUT_PATH}")


本次待审核：2,275 条；已完成：20 条
已完成 20/2,275 条
已完成 40/2,275 条
已完成 60/2,275 条
已完成 80/2,275 条
已完成 100/2,275 条
已完成 120/2,275 条
已完成 140/2,275 条
已完成 160/2,275 条
已完成 180/2,275 条
已完成 200/2,275 条
已完成 220/2,275 条
已完成 240/2,275 条
已完成 260/2,275 条
已完成 280/2,275 条
已完成 300/2,275 条
已完成 320/2,275 条
已完成 340/2,275 条
已完成 360/2,275 条
已完成 380/2,275 条
已完成 400/2,275 条
已完成 420/2,275 条
已完成 440/2,275 条
已完成 460/2,275 条
已完成 480/2,275 条
已完成 500/2,275 条
已完成 520/2,275 条
已完成 540/2,275 条
已完成 560/2,275 条
已完成 580/2,275 条
已完成 600/2,275 条
已完成 620/2,275 条
已完成 640/2,275 条
已完成 660/2,275 条
已完成 680/2,275 条
已完成 700/2,275 条
已完成 720/2,275 条
已完成 740/2,275 条
已完成 760/2,275 条
已完成 780/2,275 条
已完成 800/2,275 条
已完成 820/2,275 条
已完成 840/2,275 条
已完成 860/2,275 条
已完成 880/2,275 条
已完成 900/2,275 条
已完成 920/2,275 条
已完成 940/2,275 条
已完成 960/2,275 条
已完成 980/2,275 条
已完成 1,000/2,275 条
已完成 1,020/2,275 条
已完成 1,040/2,275 条
已完成 1,060/2,275 条
已完成 1,080/2,275 条
已完成 1,100/2,275 条
已完成 1,120/2,275 条
已完成 1,140/2,275 条
已完成 1,160/2,275 条
已完成 1,180/2,275 条
已完成 1,200/2,275 条

## 第 6 步：汇总分类结果


In [20]:
from collections import Counter

with OUTPUT_PATH.open("r", encoding="utf-8-sig", newline="") as source:
    screened = list(csv.DictReader(source))

print(f"已完成审核：{len(screened):,} 条")
print(Counter(row["llm_label"] for row in screened))

possible_names = [
    row for row in screened
    if row["llm_label"] in {"具体店名", "平台/线上商家名"}
]
print(f"可能用于后续补名的候选数：{len(possible_names):,}")


已完成审核：2,295 条
Counter({'泛称或句子片段': 1381, '具体店名': 693, '不确定': 159, '平台/线上商家名': 62})
可能用于后续补名的候选数：755


## 报告表述建议

> 对票数恰好为 2 的候选名称，采用大语言模型进行名称语义筛选，将候选划分为具体店名、平台/线上商家名、泛称或句子片段和不确定四类。该步骤仅依据候选名称的语义特征进行清洗，不直接证明候选名称与 restId 的真实对应关系。
